# **GPT2 Decoding an Sampling Methods**

This code is freely adapted from "Natural Language Processing with Transformers" (O'Reilly ed.)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "gpt2-xl"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## **1. Greedy Search Decoding**

In [ ]:
import pandas as pd

input_txt = "Transformers are the"
input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].to(device)
iterations_table = []
n_steps = 5
n_choices_per_step = 4
with torch.no_grad():
    for _ in range(n_steps):
        iteration_row = dict()
        iteration_row["Input"] = tokenizer.decode(input_ids[0])
        output = model(input_ids=input_ids)   # output contains the logits for all the tokens of the sequence.

        # Get the logits of the last position of the sequence for all the tokens of the vocabulary => next_token_logits shape: (vocabulary_size)
        next_token_logits = output.logits[0, -1, :]   # output.logits shape: (batch_size, sequence_length, vocabulary_size)

        # Apply softmax to logits
        next_token_probs = torch.softmax(next_token_logits, dim=-1)   # next_token_probs shape: (vocabulary_size)

        # Sort the token IDs by their probabilities, in descending order
        sorted_ids = torch.argsort(next_token_probs, dim=-1, descending=True)   # sorted_ids shape: (vocabulary_size)

        # Store tokens with highest probabilities (n_choices_per_step)
        for choice_id in range(n_choices_per_step):
            token_id = sorted_ids[choice_id]
            token_prob = next_token_probs[token_id].cpu().numpy()
            token_choice = (
              f"{tokenizer.decode(token_id)} ({100 * token_prob:.2f}%)"
            )
            iteration_row[f"Choice {choice_id+1}"] = token_choice   # Add string to dictionary

        # Append predicted next token to the input sequence (input_ids)
        input_ids = torch.cat([input_ids, sorted_ids[None, 0, None]], dim=-1)
        iterations_table.append(iteration_row)

pd.DataFrame(iterations_table)

,Input,Choice 1,Choice 2,Choice 3,Choice 4
0,Transformers are the,most (8.53%),only (4.96%),best (4.65%),Transformers (4.37%)
1,Transformers are the most,popular (16.78%),powerful (5.37%),common (4.96%),famous (3.72%)
2,Transformers are the most popular,toy (10.63%),toys (7.23%),Transformers (6.60%),of (5.46%)
3,Transformers are the most popular toy,line (34.38%),in (18.20%),of (11.71%),brand (6.10%)
4,Transformers are the most popular toy line,in (46.28%),of (15.09%),", (4.94%)",on (4.40%)


## **2. Sampling Methods: Top-k and Nuclues**

In [ ]:
# Initial prompt
prompt = (
    "Transformers are the foundational deep learning models that leverage the "
    "attention mechanism to process sequential data, allowing them to focus on "
    "different parts of an input sequence when making predictions. They are"
)

# Tokenization of initial prompt
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
# Create the attention_mask: the mask is set to 1 for each token of the prompt, indicating that all tokens should be considered during generation (no padding).
attention_mask = torch.ones(input_ids.shape, device=device)

# Perform con top-k sampling
output_top_k = model.generate(
    input_ids,
    attention_mask=attention_mask,  # Adding attention_mask
    max_length=50,                  # Maximum length of generation
    do_sample=True,                 # Enable sampling
    top_k=50,                       # Top-k sampling: consider only the 50 tokens with highest probability
    temperature=0.5                 # Temperature parameter
)

# Generazione con nucleus sampling (top-p)
output_top_p = model.generate(
    input_ids,
    attention_mask=attention_mask,  # Aggiunta della attention_mask
    max_length=50,
    do_sample=True,
    top_p=0.8,              # Nucleus sampling: consider the token subset with cumulative probability distribution >= 0.8
    temperature=0.5
)

# Decoding and display generated results
output_text_top_k = tokenizer.decode(output_top_k[0], skip_special_tokens=True)
output_text_top_p = tokenizer.decode(output_top_p[0], skip_special_tokens=True)

print("Top-k Sampling Output:\n\n", output_text_top_k)
print("\nNucleus Sampling Output:\n\n", output_text_top_p)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Top-k Sampling Output:

 Transformers are the foundational deep learning models that leverage the attention mechanism to process sequential data, allowing them to focus on different parts of an input sequence when making predictions. They are also the most widely used deep learning models in the world.

The

Nucleus Sampling Output:

 Transformers are the foundational deep learning models that leverage the attention mechanism to process sequential data, allowing them to focus on different parts of an input sequence when making predictions. They are used to train neural networks that can learn to recognize objects, recognize speech,
